# Stock Trading RL Environment: Training Notebook

**Meta PyTorch OpenEnv Hackathon | Theme 3 (World Modeling) + Theme 4 (Self-Improvement)**

A neural world model generates novel market scenarios. An LLM agent learns to trade against it using GRPO.

**Result: Base model 0.300 -> SFT 0.417 -> GRPO 0.537 (79% improvement)**

| Resource | Link |
|----------|------|
| Live Environment | [HF Space](https://huggingface.co/spaces/sarthakbiswas/stock-trader-env) |
| Best Model | [GRPO neural (0.537)](https://huggingface.co/sarthakbiswas/stock-trader-grpo-neural-model) |
| Market Data | [264K rows, 109 stocks](https://huggingface.co/datasets/sarthakbiswas/stock-trader-market-data) |
| Blog Post | [BLOG.md](https://huggingface.co/spaces/sarthakbiswas/stock-trader-env/blob/main/BLOG.md) |
| GitHub | [Repository](https://github.com/sarthakbiswas97/stock-trader-env) |

**Stack:** OpenEnv + TRL (GRPOTrainer) + Unsloth + PyTorch

**NOTE:** This notebook runs best with a **T4 GPU**. Go to Runtime > Change runtime type > T4 GPU.
Full training runs (300 steps GRPO, 352 steps SFT) were done on RTX 4090. Training curves and logs from those full runs are in the [GitHub repo README](https://github.com/sarthakbiswas97/stock-trader-env) and [results/ directory](https://github.com/sarthakbiswas97/stock-trader-env/tree/main/results).

## 1. Setup

Install dependencies and download market data + world model from HuggingFace Hub.

In [ ]:
%%capture
!pip install -q openenv-core gymnasium torch pandas numpy matplotlib
!pip install -q huggingface_hub datasets mlflow
!pip install -q unsloth

import os
if not os.path.exists("stock-trader-env"):
    !git clone -q https://github.com/sarthakbiswas97/stock-trader-env.git

os.chdir("stock-trader-env")
!git checkout -q main
!git pull -q origin main

from huggingface_hub import hf_hub_download
from pathlib import Path
from datasets import load_dataset

ohlcv_dir = Path("data/ohlcv")
if not ohlcv_dir.exists() or not any(ohlcv_dir.glob("*.csv")):
    ds = load_dataset("sarthakbiswas/stock-trader-market-data")
    ohlcv_dir.mkdir(parents=True, exist_ok=True)
    macro_dir = Path("data/macro")
    macro_dir.mkdir(parents=True, exist_ok=True)
    for symbol, group in ds["ohlcv"].to_pandas().groupby("symbol"):
        group.drop(columns=["symbol", "data_type"]).to_csv(ohlcv_dir / f"{symbol}_daily.csv", index=False)
    for name, group in ds["macro"].to_pandas().groupby("symbol"):
        group.drop(columns=["symbol", "data_type"]).to_csv(macro_dir / f"{name}_daily.csv", index=False)

Path("checkpoints/world_model").mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="sarthakbiswas/stock-trader-market-data",
    filename="best_transformer.pt",
    repo_type="dataset",
    local_dir="checkpoints/world_model",
)

print(f"Market data: {len(list(ohlcv_dir.glob('*.csv')))} stocks")
print("World model checkpoint downloaded")
print("Setup complete")

## 2. The Environment: What the Agent Sees (Live)

The agent receives text observations with technical indicators each day. Not raw numbers.

In [ ]:
import sys
sys.path.insert(0, ".")

from training.gym_wrapper import StockTradingGymEnv

env = StockTradingGymEnv(task_id="single_stock", seed=42, obs_mode="text", simulator_mode="replay")
obs, info = env.reset()

print("=" * 60)
print("OBSERVATION (what the LLM agent sees each day)")
print("=" * 60)
print(obs[:1000])
print("...")
print()
print(f"Portfolio value: Rs{info['portfolio_value']:,.0f}")
print(f"Cash: Rs{info['cash']:,.0f}")
print(f"Task: {info['task_id']}, Day 1/{info['total_days']}")
env.close()

## 3. Neural World Model (Live)

The causal transformer (1.22M params) generates realistic OHLCV prices. Every seed produces a unique episode. The agent can never memorize.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from server.neural_simulator import NeuralSimulator

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for idx, seed in enumerate([42, 99, 7]):
    sim = NeuralSimulator(task_id="single_stock", seed=seed, temperature=1.0)
    sim.reset()
    symbol = sim.symbols[0]

    gen_df = sim._generated[symbol]
    lookback = 100
    episode_prices = gen_df["close"].iloc[lookback:].values
    seed_prices = gen_df["close"].iloc[lookback-20:lookback].values

    ax = axes[idx]
    ax.plot(range(-20, 0), seed_prices, "b-", alpha=0.5, label="Real (seed)")
    ax.plot(range(0, len(episode_prices)), episode_prices, "r-", linewidth=2, label="Generated")
    ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(f"Neural Env (seed={seed})")
    ax.set_xlabel("Day")
    ax.set_ylabel("Price (Rs)")
    ax.legend(fontsize=8)

plt.suptitle("3 different seeds = 3 different market scenarios. Can't memorize.", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nWorld Model: {sum(p.numel() for p in sim._model.parameters()):,} parameters")
print(f"Volatility ratio: 0.94x real markets")
print(f"Training time: 7 minutes on single GPU")

## 4. Load Trained Model and Run Live Episode (GPU Required)

Load our best GRPO model from HF Hub. Run a full 20-day episode against the neural environment. Every action is generated live.

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU required. Go to Runtime > Change runtime type > T4 GPU"

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("Loading GRPO model (best: 0.537 on neural env)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    "sarthakbiswas/stock-trader-grpo-neural-model",
    max_seq_length=1024,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
FastLanguageModel.for_inference(model)
print("Model loaded.")

SYSTEM_PROMPT = (
    "You are an expert stock trader operating in the Indian equity market.\n"
    "You receive daily market observations with technical indicators and must decide on a trading action.\n\n"
    "Rules:\n"
    "- Respond with EXACTLY one action on the last line\n"
    "- Valid actions: HOLD, BUY, SELL, BUY <SYMBOL>, SELL <SYMBOL>, BUY <SYMBOL> <FRACTION>\n"
    "- Before your action, briefly explain your reasoning in 1-2 sentences inside <think> tags\n\n"
    "Example response:\n"
    "<think>RSI is 25 (oversold) and MACD just turned bullish. Good entry point.</think>\n"
    "BUY RELIANCE 0.5"
)

def run_inference(observation):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Here is today's market data:\n\n{observation}\n\nWhat is your trading action?"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                                 do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\nReady. Running live episode next.")

In [ ]:
import re

env = StockTradingGymEnv(task_id="single_stock", seed=55, obs_mode="text", simulator_mode="neural")
obs, info = env.reset()
initial_value = info["portfolio_value"]

print(f"LIVE EPISODE: Agent vs Neural Environment (seed=55)")
print(f"Starting capital: Rs{initial_value:,.0f}, 20 trading days")
print("=" * 70)

for day in range(20):
    response = run_inference(obs)

    # Parse action from response
    action = "HOLD"
    for line in reversed(response.strip().split("\n")):
        line_upper = line.strip().upper()
        if line_upper.startswith(("BUY", "SELL", "HOLD")):
            match = re.match(r"(BUY|SELL|HOLD).*", line.strip(), re.IGNORECASE)
            if match:
                action = line.strip()
            break

    obs, reward, done, _, info = env.step(action)

    # Show reasoning + action for key days
    if day < 5 or day >= 18 or "BUY" in action.upper() or "SELL" in action.upper():
        think = ""
        if "<think>" in response and "</think>" in response:
            think = response.split("<think>")[1].split("</think>")[0][:80]
        print(f"Day {day+1:2d} | {action:22s} | Rs{info['portfolio_value']:>10,.0f} | {think}...")

    if done:
        break

final_value = info["portfolio_value"]
episode_return = (final_value - initial_value) / initial_value * 100
print("=" * 70)
print(f"Score: {info['score']:.3f}")
print(f"Return: {episode_return:+.2f}%")
print(f"Portfolio: Rs{initial_value:,.0f} -> Rs{final_value:,.0f}")
env.close()

## 5. Live GRPO Training (10 steps)

Actual GRPO training using TRL GRPOTrainer + Unsloth. This runs 10 steps to demonstrate the pipeline works end-to-end. Full training (300 steps, ~1 hour on RTX 4090) produced the 0.537 model. Training curves from the full run are in the [GitHub repo](https://github.com/sarthakbiswas97/stock-trader-env/tree/main/results).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import gc

# Free the inference model from Cell 4 to make room for training
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("Freed inference model VRAM.")
except:
    pass

# Stub packages that TRL imports but we don't use
import types as _types
import sys as _sys
for _pkg in ["mergekit", "mergekit.config", "llm_blender", "weave"]:
    _sys.modules.setdefault(_pkg, _types.ModuleType(_pkg))

import transformers.utils.hub as _hub
if not hasattr(_hub, "TRANSFORMERS_CACHE"):
    from huggingface_hub.constants import HF_HUB_CACHE
    _hub.TRANSFORMERS_CACHE = HF_HUB_CACHE

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import json, math, re
from pathlib import Path
from huggingface_hub import hf_hub_download

# Load SFT model (starting point for GRPO)
print("Loading SFT v3 model...")
grpo_model, grpo_tokenizer = FastLanguageModel.from_pretrained(
    "sarthakbiswas/stock-trader-sft-v3-model",
    max_seq_length=1024,
    load_in_4bit=True,
)
grpo_tokenizer = get_chat_template(grpo_tokenizer, chat_template="qwen-2.5")

from peft import PeftModel
if not isinstance(grpo_model, PeftModel):
    grpo_model = FastLanguageModel.get_peft_model(
        grpo_model, r=64, lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none",
    )

# Download prompts collected from neural env episodes
print("Downloading GRPO prompts...")
Path("/tmp/grpo-data").mkdir(exist_ok=True)
hf_hub_download(
    repo_id="sarthakbiswas/stock-trader-market-data",
    filename="grpo_improved_prompts.jsonl",
    repo_type="dataset",
    local_dir="/tmp/grpo-data",
)
prompts_data = [json.loads(l) for l in open("/tmp/grpo-data/grpo_improved_prompts.jsonl")]
train_dataset = Dataset.from_list(prompts_data[:200])  # Use subset for demo
print(f"Loaded {len(train_dataset)} prompts")

# Reward functions (same as full training)
def format_gate(completions, **kwargs):
    """Penalty-only: 0 if valid format, -1 if invalid."""
    rewards = []
    for c in completions:
        text = c if isinstance(c, str) else "".join(m.get("content", "") if isinstance(m, dict) else str(m) for m in c)
        has_think = "<think>" in text and "</think>" in text
        lines = text.strip().split("\n")
        last = lines[-1].strip().upper() if lines else ""
        has_action = last.startswith(("BUY", "SELL", "HOLD"))
        rewards.append(0.0 if (has_think and has_action) else -1.0)
    return rewards

def trading_reward(completions, **kwargs):
    """Blended reward: 30% step-level + 70% episode-level."""
    current_prices = kwargs.get("current_price", [])
    next_1d = kwargs.get("next_price_1d", [])
    has_positions = kwargs.get("has_position", [])
    episode_returns = kwargs.get("episode_return", [])

    rewards = []
    for i, c in enumerate(completions):
        text = c if isinstance(c, str) else "".join(m.get("content", "") if isinstance(m, dict) else str(m) for m in c)
        action = "HOLD"
        for line in reversed(text.strip().split("\n")):
            m = re.match(r"(BUY|SELL|HOLD)", line.strip().upper())
            if m:
                action = m.group(1)
                break

        cur = current_prices[i] if i < len(current_prices) else 0.0
        nxt = next_1d[i] if i < len(next_1d) else cur
        has_pos = has_positions[i] if i < len(has_positions) else False
        ep_ret = episode_returns[i] if i < len(episode_returns) else 0.0

        if cur > 0:
            change = (nxt - cur) / cur
            vol = max(abs(change), 0.005)
            z = change / vol
            step_r = math.tanh(z * 1.5)
            if action == "BUY" and not has_pos:
                step_r = step_r if z >= 0 else step_r * 1.5
            elif action == "SELL" and has_pos:
                step_r = -step_r if z <= 0 else -step_r * 1.5
            elif action == "HOLD":
                step_r = step_r * 0.3 if has_pos else -abs(step_r) * 0.2
            else:
                step_r = -0.3
        else:
            step_r = 0.0

        ep_r = math.tanh(ep_ret * 100 * 0.4)
        rewards.append(float(0.3 * step_r + 0.7 * ep_r))
    return rewards

# Detect T4 (no bf16) vs Ampere+ (bf16)
use_bf16 = torch.cuda.get_device_capability()[0] >= 8

# GRPO config (10 steps for demo)
config = GRPOConfig(
    output_dir="/tmp/grpo-demo",
    num_generations=4,
    max_completion_length=200,
    beta=0.05,
    learning_rate=5e-7,
    max_steps=10,
    logging_steps=2,
    save_steps=999,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    bf16=use_bf16,
    fp16=not use_bf16,
    seed=42,
    temperature=0.8,
    loss_type="grpo",
    scale_rewards="batch",
)

trainer = GRPOTrainer(
    model=grpo_model,
    args=config,
    train_dataset=train_dataset,
    reward_funcs=[format_gate, trading_reward],
    processing_class=grpo_tokenizer,
)

print(f"\nStarting GRPO training (10 steps demo, {'bf16' if use_bf16 else 'fp16'})...")
print("Full training: 300 steps on RTX 4090, ~1 hour. Curves in GitHub repo.")
print()
trainer.train()
print("\nGRPO training complete (10 steps demo).")
print("Full training logs and reward curves: https://github.com/sarthakbiswas97/stock-trader-env/tree/main/results")

In [ ]:
# Plot live training metrics from the 10-step demo run
import matplotlib.pyplot as plt
import numpy as np

log_history = trainer.state.log_history
train_logs = [l for l in log_history if "loss" in l and "kl" in l]

if train_logs:
    steps = [l["step"] for l in train_logs]
    loss = [l["loss"] for l in train_logs]
    kl = [l["kl"] for l in train_logs]
    reward = [l["reward"] for l in train_logs]
    trading_r = [l.get("rewards/trading_reward/mean", 0) for l in train_logs]

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.patch.set_facecolor("white")
    for ax in axes:
        ax.set_facecolor("#fafafa")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].plot(steps, loss, "b-o", linewidth=2, markersize=6)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss", fontweight="bold")

    axes[1].plot(steps, kl, "g-o", linewidth=2, markersize=6)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("KL Divergence")
    axes[1].set_title("KL Divergence", fontweight="bold")

    axes[2].plot(steps, reward, "r-o", linewidth=2, markersize=6)
    axes[2].axhline(y=0, color="gray", linewidth=0.8, alpha=0.5)
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Total Reward")
    axes[2].set_title("Total Reward", fontweight="bold")

    axes[3].plot(steps, trading_r, "m-o", linewidth=2, markersize=6)
    axes[3].axhline(y=0, color="gray", linewidth=0.8, alpha=0.5)
    axes[3].set_xlabel("Step")
    axes[3].set_ylabel("Trading Reward")
    axes[3].set_title("Trading Reward", fontweight="bold")

    fig.suptitle("Live GRPO Training Metrics (10 steps demo)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"\nStep {steps[-1]}: loss={loss[-1]:.4f}, KL={kl[-1]:.4f}, reward={reward[-1]:.3f}, trading={trading_r[-1]:.3f}")
    print(f"\nThis is a 10-step demo. Full 300-step training curves are in Cell 6 below.")
    print(f"Full logs: https://github.com/sarthakbiswas97/stock-trader-env/tree/main/results")
else:
    print("No training logs found. Run the GRPO training cell above first.")

## 6. Training Evidence from Full Runs

These plots are from the actual 300-step GRPO training and 352-step SFT training runs on RTX 4090. Source: `results/sft_v3_training_log.json` and `results/grpo_neural_training_log.json` in the [repo](https://github.com/sarthakbiswas97/stock-trader-env/tree/main/results).

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open("results/sft_v3_training_log.json") as f:
    sft_log = json.load(f)
with open("results/grpo_neural_training_log.json") as f:
    grpo_log = json.load(f)

C_SFT, C_GRPO, C_FAIL, C_GRAY = "#3498db", "#2ecc71", "#e74c3c", "#95a5a6"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor("white")
for ax in axes.flat:
    ax.set_facecolor("#fafafa")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Panel A: Learning Curve
models = ["Base\nDeepSeek 7B", "SFT v3\n(distilled)", "GRPO Neural\n(BEST)"]
static = [0.300, 0.399, 0.470]
neural = [0.298, 0.417, 0.537]
x = np.arange(len(models))
w = 0.32
b1 = axes[0,0].bar(x-w/2, static, w, label="Static Env", color=C_SFT, edgecolor="white", linewidth=1.5)
b2 = axes[0,0].bar(x+w/2, neural, w, label="Neural Env", color=C_GRPO, edgecolor="white", linewidth=1.5)
axes[0,0].axhline(y=0.300, color=C_FAIL, linestyle="--", linewidth=1.2, alpha=0.5)
axes[0,0].text(2.45, 0.305, "HOLD floor", fontsize=7.5, color=C_FAIL, alpha=0.7)
for bar in b1:
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
        f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold", color=C_SFT)
for bar in b2:
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
        f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold", color="#1a8a4a")
axes[0,0].annotate("", xy=(2.16,0.53), xytext=(0.16,0.30),
    arrowprops=dict(arrowstyle="->", color=C_GRAY, lw=2, connectionstyle="arc3,rad=0.15"))
axes[0,0].text(1.1, 0.48, "+79%", fontsize=11, fontweight="bold", color=C_GRAY, ha="center")
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(models, fontsize=10)
axes[0,0].set_ylabel("Score")
axes[0,0].set_ylim(0.2, 0.62)
axes[0,0].set_title("Result: Base 0.300 -> GRPO 0.537", fontweight="bold")
axes[0,0].legend(fontsize=9, loc="upper left")

# Panel B: SFT Loss
steps = [e["step"] for e in sft_log]
losses = [e["loss"] for e in sft_log]
axes[0,1].plot(steps, losses, color=C_SFT, linewidth=2.5)
axes[0,1].fill_between(steps, losses, alpha=0.08, color=C_SFT)
axes[0,1].axvline(x=200, color=C_GRPO, linestyle="--", alpha=0.7)
best_idx = next(i for i, e in enumerate(sft_log) if e["step"] == 200)
axes[0,1].plot(200, losses[best_idx], "o", color=C_GRPO, markersize=10, zorder=5)
axes[0,1].annotate("Best checkpoint\nscore 0.399", xy=(200, losses[best_idx]),
    xytext=(120, losses[best_idx]+0.15), fontsize=8.5, fontweight="bold", color=C_GRPO,
    arrowprops=dict(arrowstyle="->", color=C_GRPO, lw=1.2))
axes[0,1].set_xlabel("Training Step")
axes[0,1].set_ylabel("Loss")
axes[0,1].set_title("SFT Training Loss (lower loss != better trading)", fontweight="bold")

# Panel C: KL Divergence
g_steps = [e["step"] for e in grpo_log]
kl = [e["kl"] for e in grpo_log]
axes[1,0].plot(g_steps, kl, color=C_GRPO, linewidth=2)
axes[1,0].fill_between(g_steps, kl, alpha=0.1, color=C_GRPO)
axes[1,0].axhline(y=3.0, color=C_FAIL, linestyle="--", linewidth=1.5, alpha=0.7)
axes[1,0].text(155, 3.15, "Previous GRPO v3 collapsed here (KL=4.2)", fontsize=8, color=C_FAIL, fontweight="bold", ha="center")
axes[1,0].fill_between([0, 300], [0, 0], [0.5, 0.5], alpha=0.05, color=C_GRPO)
axes[1,0].text(150, 0.05, "Safe zone (beta=0.05)", fontsize=8, color=C_GRPO, alpha=0.7, ha="center")
axes[1,0].set_xlabel("Training Step")
axes[1,0].set_ylabel("KL Divergence")
axes[1,0].set_title("GRPO KL Divergence (stayed under 0.35)", fontweight="bold")
axes[1,0].set_ylim(-0.05, 3.8)

# Panel D: Trading Reward
r_mean = [e["rewards/trading_reward/mean"] for e in grpo_log]
r_std = [e["rewards/trading_reward/std"] for e in grpo_log]
axes[1,1].fill_between(g_steps, [m-s for m,s in zip(r_mean,r_std)],
    [m+s for m,s in zip(r_mean,r_std)], alpha=0.15, color=C_GRPO, label="Reward std")
axes[1,1].plot(g_steps, r_mean, color=C_GRPO, linewidth=2, label="Trading reward")
if len(r_mean) >= 5:
    smoothed = np.convolve(r_mean, np.ones(5)/5, mode="valid")
    axes[1,1].plot(g_steps[4:], smoothed, color="#27ae60", linewidth=2.5, alpha=0.8, label="Trend (5-step avg)")
axes[1,1].axhline(y=0, color=C_GRAY, linewidth=0.8, alpha=0.5)
axes[1,1].set_xlabel("Training Step")
axes[1,1].set_ylabel("Trading Reward")
axes[1,1].set_title("GRPO Trading Reward Signal", fontweight="bold")
axes[1,1].legend(fontsize=8, loc="lower left")

fig.suptitle("Training Evidence: SFT + GRPO Against Neural Environment (Full Runs)",
    fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print(f"Improvement: Base 0.300 -> SFT 0.417 -> GRPO 0.537")
print(f"Total: +79% over base model (neural env)")
print(f"KL stayed under 0.35 (previous GRPO v3 reached 4.2 and collapsed)")
print(f"\nFull logs: https://github.com/sarthakbiswas97/stock-trader-env/tree/main/results")

## 7. Reward Design and Safeguards

In [ ]:
from server.mistake_tracker import MistakeTracker, MistakeType

print("REWARD DESIGN")
print("=" * 60)
print()
print("1. FORMAT GATE (penalty only)")
print("   Valid format: 0.0 reward (expected, not rewarded)")
print("   Invalid format: -1.0 penalty")
print()
print("2. TRADING REWARD (blended)")
print("   30% step-level: 5D composite P&L with asymmetric penalties")
print("   70% episode-level: aligned with eval grading")
print()
print("3. SIGNAL-BASED HOLD")
print("   HOLD is not free. Missed RSI < 25: -0.15 penalty")
print("   Justified patience (neutral indicators): +0.01")
print()
print("4. MISTAKE TRACKER (7 error types)")
for mt in MistakeType:
    print(f"   - {mt.value}")
print()
print("5. ANTI-HOLD COLLAPSE")
print("   Rolling window: if HOLD > 75%, extra penalty")
print("   Bonus for BUY/SELL when HOLD dominates")

## 8. Full Scoreboard: 15 Model Iterations

Every model trained, in order. The failures matter as much as the successes. Full story in [BLOG.md](https://huggingface.co/spaces/sarthakbiswas/stock-trader-env/blob/main/BLOG.md).

In [ ]:
scoreboard = [
    ("DeepSeek 7B (base)",    "0.300", "0.298", "No training"),
    ("SFT v1",               "0.300", "-",     "91% HOLD, data imbalance"),
    ("SFT v2",               "0.398", "-",     "Template reasoning"),
    ("GRPO v1",              "~0.300","-",     "Collapsed to HOLD"),
    ("GRPO v2",              "0.326", "-",     "84% reward from formatting"),
    ("GRPO v2.1",            "0.395", "-",     "Reward-aligned, bimodal"),
    ("GRPO v3",              "0.301", "-",     "KL catastrophe (4.2)"),
    ("SFT v3 step-352",      "0.383", "-",     "Overtrained"),
    ("SFT v3 step-200",      "0.399", "0.417", "Distilled reasoning, best SFT"),
    ("RAFT (from base)",     "0.300", "0.300", "Wrong starting point"),
    ("RAFT v2 (from SFT v3)","0.360", "0.399", "640 winners"),
    ("GRPO neural",          "0.470", "0.537", "BEST: RL against neural env"),
    ("GRPO v3.1",            "0.418", "0.310", "Improved env (harder)"),
    ("GRPO v3.2 ckpt-400",   "-",     "0.485", "Improved env, HOLD% 95->85%"),
    ("GRPO v3.3",            "-",     "0.416", "Lower beta, didn't converge"),
]

print(f"{'#':>2}  {'Model':<25} {'Static':>7} {'Neural':>7}  {'Notes'}")
print("-" * 75)
for i, (name, static, neural, notes) in enumerate(scoreboard, 1):
    marker = "  <<" if "BEST" in notes else ""
    print(f"{i:>2}  {name:<25} {static:>7} {neural:>7}  {notes}{marker}")

print()
print("15 iterations. 4 crashes. 1 breakthrough.")
print("Best model: https://huggingface.co/sarthakbiswas/stock-trader-grpo-neural-model")

## 9. Summary

In [ ]:
print("""
ARCHITECTURE
============

Real Market Data (264K rows, 109 NIFTY stocks, 2015-2026)
        |
        v
Causal Transformer World Model (1.22M params, 4 layers, MDN output)
  Volatility: 0.94x reality. Trained in 7 min.
        |
        v
Generated OHLCV --> Feature Engine --> Text Observation
  (RSI, MACD, Bollinger, trend, momentum, regime, volume)
        |
        v
LLM Agent (DeepSeek-R1 7B, QLoRA r=16)
  Reads text, outputs: <think>reasoning</think> ACTION
        |
        v
Environment Grader (verifiable score 0.0-1.0)


TRAINING PIPELINE
=================

1. SFT: 10K reverse-distilled examples      0.300 -> 0.417
2. RAFT: 640 winning episodes                (slight degradation)
3. GRPO: RL against neural env, 300 steps    0.417 -> 0.537

Total: 79% improvement over base model.
15 iterations. 4 crashes. Lessons in every failure.
""")

print("Live environment: https://huggingface.co/spaces/sarthakbiswas/stock-trader-env")
print("Best model:       https://huggingface.co/sarthakbiswas/stock-trader-grpo-neural-model")
print("Market data:      https://huggingface.co/datasets/sarthakbiswas/stock-trader-market-data")
print("Blog post:        https://huggingface.co/spaces/sarthakbiswas/stock-trader-env/blob/main/BLOG.md")
print("GitHub:           https://github.com/sarthakbiswas97/stock-trader-env")
print()
print("Built by Sarthak Biswas for the Meta PyTorch OpenEnv Hackathon, April 2026.")